# SLAP2 DMD Super-Stack Merge QC

This notebook is for developing and inspecting SLAP2 GUI `*-REFERENCE.tif` DMD stack merging. It supports two workflows:

1. **Full reference stacks** with SLAP2 JSON metadata preserved in each TIFF page description.
2. **Fiji/ImageJ substacks** where the original JSON was stripped. For these, provide the z-start/z-spacing and `dmdPixel2SampleTransform` manually.

The current intended merge is:

```text
DMD1 superficial stack + DMD2 deeper stack -> common sample-coordinate super stack
```

DMD1 is assumed superficial by default, larger z values are deeper, and the DMD1/DMD2 overlap is inferred from z positions.


In [ ]:
from pathlib import Path
from typing import Optional
import sys
import json
import warnings

from IPython.display import display, HTML


def find_repo_root(start: Optional[Path] = None) -> Path:
    if start is None:
        start = Path.cwd()
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "slap2_volume_align").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find repo root. Expected pyproject.toml and src/slap2_volume_align.\n"
        f"Started from: {start}"
    )


REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Repo root: {REPO_ROOT}")
print(f"Python: {sys.version}")
display(HTML("<style>.container { width:100% !important; }</style>"))
warnings.filterwarnings("default")


In [ ]:
import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt

from slap2_volume_align.sources.slap2.metadata import (
    read_reference_stack_spec,
    make_manual_reference_stack_spec,
    offset_reference_stack_z,
)
from slap2_volume_align.sources.slap2.geometry import (
    compute_output_grid,
    infer_z_overlap_um,
    stack_corners_sample_xy,
)
from slap2_volume_align.sources.slap2.qc import plot_dmd_footprints
from slap2_volume_align.sources.slap2.merge import (
    Slap2MergeConfig,
    merge_dmd_reference_stacks,
    merge_dmd_reference_stack_specs,
)

print("Imports OK")


## Configure paths

Use the full original `*-REFERENCE.tif` files when possible. If you are testing with Fiji-created overlap substacks, set `USE_MANUAL_SUBSTACK_METADATA = True`.


In [ ]:
# Full SLAP2 GUI reference stacks. These should preserve per-page JSON metadata.
DMD1_REFERENCE_TIF = Path(
    r"\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\836174\836174_2026-06-17_12-51-17\slap2\static_data\structure_volume_20260617_132712_DMD1-REFERENCE.tif"
)
DMD2_REFERENCE_TIF = Path(
    r"\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\836174\836174_2026-06-17_12-51-17\slap2\static_data\structure_volume_20260617_132712_DMD2-REFERENCE.tif"
)

# Fiji/ImageJ overlap substacks for fast testing. These often lose the SLAP2 JSON metadata.
DMD1_SUBSTACK_TIF = Path(
    r"\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\836174\836174_2026-06-17_12-51-17\slap2\static_data\structure_volume_20260617_132712_DMD1-REFERENCE_substack.tif"
)
DMD2_SUBSTACK_TIF = Path(
    r"\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\836174\836174_2026-06-17_12-51-17\slap2\static_data\structure_volume_20260617_132712_DMD2-REFERENCE_substack.tif"
)

# Set this True for the last-20/first-20 Fiji substacks.
USE_MANUAL_SUBSTACK_METADATA = False

# Axial stitch calibration for this DMD pair. The overlap diagnostic below
# estimated that DMD2 should be shifted 5 planes superficial, i.e. -7.5 µm.
# Set to 0.0 for a metadata-only baseline merge.
DMD2_Z_OFFSET_UM = -7.5

# XY residual registration should be estimated after applying the z offset.
FINE_REGISTER_OVERLAP = True

OUT_DIR = Path(
    r"\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\836174\836174_2026-06-17_12-51-17\slap2\static_data\super_stack_qc"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(OUT_DIR)


## Manual metadata for Fiji substacks

Fiji-created substacks may replace the SLAP2 JSON `ImageDescription` with ImageJ metadata. In that case, we reconstruct the minimal metadata needed for QC using the known transforms and z ranges.

For your uploaded overlap substacks, the files contained 21 pages. The default values below assume:

- DMD1 substack = full-stack pages 80–100, z = -79.875 to -49.875 µm
- DMD2 substack = full-stack pages 0–20, z = -60 to -30 µm

Adjust `DMD1_SUBSTACK_Z_START_UM` if your local substack contains exactly 20 pages instead of 21.


In [ ]:
# Transforms from the SLAP2 REFERENCE TIFF JSON.
DMD1_DMD_PIXEL_TO_SAMPLE_TRANSFORM = [
    [0.24907835903790651, -0.0008847190020129167, 0, -1.902316580004602],
    [0.0029171477472943433, 0.25063596511811065, 0, -0.63452150251753014],
    [0, 0, 1, -199.875],
    [0, 0, 0, 1],
]
DMD2_DMD_PIXEL_TO_SAMPLE_TRANSFORM = [
    [0.17567302550946895, 0.1969730316452333, 0, -25.140401931388212],
    [0.16170638488237649, -0.17629969847000435, 0, 52.244553573038651],
    [0, 0, 1, -60],
    [0, 0, 0, 1],
]

DMD1_SUBSTACK_Z_START_UM = -79.875  # last 21 planes: pages 80-100. For last 20 use -78.375.
DMD2_SUBSTACK_Z_START_UM = -60.0
Z_SPACING_UM = 1.5


## Inspect TIFF structure and build stack specs

In [ ]:
def inspect_tiff(path: Path, n_desc_chars: int = 500):
    print(f"\n== {path}")
    if not path.exists():
        print("MISSING")
        return
    with tifffile.TiffFile(path) as tif:
        print("n pages:", len(tif.pages))
        print("series:", [(s.shape, s.axes, str(s.dtype)) for s in tif.series])
        print("page 0 shape/dtype:", tif.pages[0].shape, tif.pages[0].dtype)
        desc = tif.pages[0].description or ""
        print(desc[:n_desc_chars])

if USE_MANUAL_SUBSTACK_METADATA:
    DMD1_TIF = DMD1_SUBSTACK_TIF
    DMD2_TIF = DMD2_SUBSTACK_TIF
else:
    DMD1_TIF = DMD1_REFERENCE_TIF
    DMD2_TIF = DMD2_REFERENCE_TIF

inspect_tiff(DMD1_TIF)
inspect_tiff(DMD2_TIF)


In [ ]:
if USE_MANUAL_SUBSTACK_METADATA:
    spec1 = make_manual_reference_stack_spec(
        DMD1_TIF,
        z_start_um=DMD1_SUBSTACK_Z_START_UM,
        z_spacing_um=Z_SPACING_UM,
        transform=DMD1_DMD_PIXEL_TO_SAMPLE_TRANSFORM,
        channel=1,
        acquisition_path_idx=1,
    )
    spec2 = make_manual_reference_stack_spec(
        DMD2_TIF,
        z_start_um=DMD2_SUBSTACK_Z_START_UM,
        z_spacing_um=Z_SPACING_UM,
        transform=DMD2_DMD_PIXEL_TO_SAMPLE_TRANSFORM,
        channel=1,
        acquisition_path_idx=2,
    )
else:
    spec1 = read_reference_stack_spec(DMD1_TIF)
    spec2 = read_reference_stack_spec(DMD2_TIF)

# Apply an axial DMD2 stitch correction only to the in-memory spec used for
# footprint/grid QC. The original TIFF metadata is not modified.
spec2_effective = offset_reference_stack_z(spec2, DMD2_Z_OFFSET_UM)

for name, spec in [("DMD1", spec1), ("DMD2 raw", spec2), ("DMD2 effective", spec2_effective)]:
    print(f"\n{name}")
    print("path:", spec.path)
    print("shape:", spec.n_pages, spec.image_shape, spec.dtype, spec.axes)
    print("channels:", spec.channels)
    print("z range:", spec.z_min_um, spec.z_max_um, "dz:", spec.z_spacing_um)
    print("acq path:", spec.acquisition_path_idx)

print("\nRaw Z overlap (µm):", infer_z_overlap_um(spec1, spec2))
print("Effective Z overlap (µm):", infer_z_overlap_um(spec1, spec2_effective))
print("DMD2 z offset (µm):", DMD2_Z_OFFSET_UM)


## Footprint QC

This should resemble the SLAP2 GUI view: DMD1 is approximately axis-aligned, DMD2 is rotated, and the two footprints largely overlap in sample XY.


In [ ]:
XY_RESOLUTION_UM = 0.25 if not USE_MANUAL_SUBSTACK_METADATA else 1.0  # use 1 µm for fast subset tests
Z_RESOLUTION_UM = 1.5

grid = compute_output_grid(
    [spec1, spec2_effective],
    xy_resolution_um=XY_RESOLUTION_UM,
    z_resolution_um=Z_RESOLUTION_UM,
    z_grid="first",
)
print(grid)

fig = plot_dmd_footprints(
    [spec1, spec2_effective],
    labels=["DMD1", "DMD2 effective"],
    grid=grid,
    out_path=OUT_DIR / "notebook_slap2_dmd_footprints_qc.png",
)
plt.show()


## Raw substack projections

This is useful for sanity-checking the overlap substacks before warping.


In [ ]:
def load_small_stack(path: Path) -> np.ndarray:
    arr = tifffile.imread(path)
    if arr.ndim == 2:
        arr = arr[None, ...]
    return arr.astype(np.float32, copy=False)

# This cell is safe for the overlap substacks. Avoid running on full 8-10 GB files unless you intend to load them.
if USE_MANUAL_SUBSTACK_METADATA:
    raw1 = load_small_stack(DMD1_TIF)
    raw2 = load_small_stack(DMD2_TIF)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, img, title in zip(axes, [np.nanmax(raw1, axis=0), np.nanmax(raw2, axis=0)], ["DMD1 raw max", "DMD2 raw max"]):
        lo, hi = np.nanpercentile(img, [1, 99.7])
        ax.imshow(img, cmap="gray", vmin=lo, vmax=hi)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
else:
    print("Skipping raw full-stack load. Set USE_MANUAL_SUBSTACK_METADATA=True for substack projection preview.")


## Merge test

For full reference stacks, start at native-ish `XY_RESOLUTION_UM = 0.25`. For subset tests, use `1.0` µm first for speed. Once the geometry looks right, reduce to `0.5` or `0.25`.


In [ ]:
MERGE_OUT_DIR = OUT_DIR / ("manual_substack_merge" if USE_MANUAL_SUBSTACK_METADATA else "full_reference_merge")
MERGE_OUT_DIR.mkdir(parents=True, exist_ok=True)

config = Slap2MergeConfig(
    dmd1_tif=DMD1_TIF,
    dmd2_tif=DMD2_TIF,
    out_dir=MERGE_OUT_DIR,
    channel=1,
    xy_resolution_um=XY_RESOLUTION_UM,
    z_resolution_um=Z_RESOLUTION_UM,
    z_grid="first",
    dmd2_z_offset_um=DMD2_Z_OFFSET_UM,
    z_interp_method="linear",
    output_compression=None,
    fine_register_overlap=FINE_REGISTER_OVERLAP,
    residual_registration_binning=2,
    residual_upsample_factor=10,
    residual_highpass_sigma_px=8.0,
    residual_max_shift_px=50.0,
    xy_feather_px=32.0,
    z_feather=True,
    write_intermediates=True,
    write_qc_png=True,
)

if USE_MANUAL_SUBSTACK_METADATA:
    summary = merge_dmd_reference_stack_specs(config, spec1, spec2)
else:
    summary = merge_dmd_reference_stacks(config)

print("Z registration:")
print(json.dumps(summary.get("z_registration", {}), indent=2))
print("Residual XY registration:")
print(json.dumps(summary.get("residual_registration", {}), indent=2))
print("Outputs:")
print(json.dumps(summary["outputs"], indent=2))
print("Summary JSON written to:", MERGE_OUT_DIR / "slap2_super_stack_merge_summary.json")


## Inspect merged output lazily

In [ ]:
super_stack_path = Path(summary["outputs"]["super_stack"])
print(super_stack_path)

with tifffile.TiffFile(super_stack_path) as tif:
    print("pages:", len(tif.pages))
    print("series:", [(s.shape, s.axes, str(s.dtype)) for s in tif.series])
    mid = len(tif.pages) // 2
    sample_pages = [0, mid, len(tif.pages) - 1]
    fig, axes = plt.subplots(1, len(sample_pages), figsize=(15, 5))
    for ax, idx in zip(axes, sample_pages):
        img = tif.pages[idx].asarray()
        finite = np.isfinite(img)
        if finite.any():
            lo, hi = np.nanpercentile(img[finite], [1, 99.7])
        else:
            lo, hi = 0, 1
        ax.imshow(img, cmap="gray", vmin=lo, vmax=hi)
        ax.set_title(f"z index {idx}")
        ax.axis("off")
    plt.tight_layout()


## Overlap z-offset diagnostic

Use this after a merge that writes `dmd1_warped_ch1.tif` and `dmd2_warped_ch1.tif`. It compares DMD1 and DMD2 warped planes around the seam and estimates the best integer-plane DMD2 lag. For the current dataset this diagnostic previously estimated `DMD2_Z_OFFSET_UM = -7.5`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import tifffile
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

MERGE_DIR = MERGE_OUT_DIR
summary_path = MERGE_DIR / "slap2_super_stack_merge_summary.json"

with open(summary_path, "r") as f:
    merge_summary = json.load(f)

dmd1_path = Path(merge_summary["outputs"]["dmd1_warped"])
dmd2_path = Path(merge_summary["outputs"]["dmd2_warped"])

z0 = float(merge_summary["output_grid"]["z_min_um"])
dz = float(merge_summary["output_grid"]["z_resolution_um"])
z_overlap = merge_summary["z_overlap_um"]

with tifffile.TiffFile(dmd1_path) as tif:
    n_z = len(tif.pages)

z_um = z0 + np.arange(n_z) * dz

# Search beyond the nominal/corrected overlap. Increase margin if the diagnostic
# suggests a large axial offset or if morphology spans far across the seam.
margin_um = 18.0
candidate_idx = np.where(
    (z_um >= z_overlap[0] - margin_um) &
    (z_um <= z_overlap[1] + margin_um)
)[0]

print("Current configured DMD2 z offset:", merge_summary.get("z_registration", {}).get("dmd2_z_offset_um"))
print("Effective overlap:", z_overlap)
print("candidate planes:")
for idx in candidate_idx:
    print(f"Fiji plane {idx + 1:03d}, zero-index {idx:03d}, z = {z_um[idx]:.3f} um")


def read_plane(path: Path, idx: int) -> np.ndarray:
    with tifffile.TiffFile(path) as tif:
        return tif.pages[int(idx)].asarray().astype(np.float32, copy=False)


def preprocess_for_zcorr(img: np.ndarray, highpass_sigma_px: float = 12.0) -> np.ndarray:
    """High-pass and robust-normalize one plane for z similarity."""
    img = img.astype(np.float32, copy=False)
    img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)
    hp = img - gaussian_filter(img, highpass_sigma_px)

    vals = hp[np.isfinite(hp) & (img > 0)]
    if vals.size < 100:
        return hp * 0.0

    med = np.median(vals)
    mad = np.median(np.abs(vals - med)) + 1e-6
    hp = (hp - med) / (1.4826 * mad)
    hp = np.clip(hp, -5, 5)
    return hp.astype(np.float32)


def ncc(a: np.ndarray, b: np.ndarray, valid_mask: np.ndarray) -> float:
    vals_a = a[valid_mask]
    vals_b = b[valid_mask]

    if vals_a.size < 1000:
        return np.nan

    vals_a = vals_a - np.mean(vals_a)
    vals_b = vals_b - np.mean(vals_b)

    denom = np.sqrt(np.sum(vals_a ** 2) * np.sum(vals_b ** 2))
    if denom <= 0:
        return np.nan

    return float(np.sum(vals_a * vals_b) / denom)


# Lazily read only candidate planes instead of loading full warped volumes.
dmd1_prep = {}
dmd2_prep = {}
dmd1_valid = {}
dmd2_valid = {}

for idx in candidate_idx:
    img1 = read_plane(dmd1_path, int(idx))
    img2 = read_plane(dmd2_path, int(idx))
    dmd1_prep[idx] = preprocess_for_zcorr(img1)
    dmd2_prep[idx] = preprocess_for_zcorr(img2)
    dmd1_valid[idx] = np.isfinite(img1) & (img1 > 0)
    dmd2_valid[idx] = np.isfinite(img2) & (img2 > 0)

score = np.full((len(candidate_idx), len(candidate_idx)), np.nan, dtype=np.float32)

for ii, i in enumerate(candidate_idx):
    for jj, j in enumerate(candidate_idx):
        valid = dmd1_valid[i] & dmd2_valid[j]
        score[ii, jj] = ncc(dmd1_prep[i], dmd2_prep[j], valid)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(
    score,
    origin="lower",
    aspect="auto",
    extent=[
        z_um[candidate_idx[0]],
        z_um[candidate_idx[-1]],
        z_um[candidate_idx[0]],
        z_um[candidate_idx[-1]],
    ],
)
ax.plot(
    [z_um[candidate_idx[0]], z_um[candidate_idx[-1]]],
    [z_um[candidate_idx[0]], z_um[candidate_idx[-1]]],
    "w--",
    lw=1,
    label="no residual z offset",
)
ax.set_xlabel("DMD2 effective z, um")
ax.set_ylabel("DMD1 z, um")
ax.set_title("DMD1-vs-DMD2 z similarity after XY correction")
ax.legend(loc="upper left")
plt.colorbar(im, ax=ax, label="normalized cross-correlation")
plt.tight_layout()

lags = np.arange(-10, 11)
lag_scores = []
idx_to_pos = {int(idx): pos for pos, idx in enumerate(candidate_idx)}

for lag in lags:
    vals = []
    for i in candidate_idx:
        j = int(i + lag)
        if j in idx_to_pos:
            ii = idx_to_pos[int(i)]
            jj = idx_to_pos[j]
            if np.isfinite(score[ii, jj]):
                vals.append(score[ii, jj])
    lag_scores.append(np.nanmedian(vals) if vals else np.nan)

lag_scores = np.asarray(lag_scores)
best_lag = int(lags[np.nanargmax(lag_scores)])
additional_dmd2_z_offset_um = -best_lag * dz
current_offset = float(merge_summary.get("z_registration", {}).get("dmd2_z_offset_um", 0.0) or 0.0)
recommended_total_offset = current_offset + additional_dmd2_z_offset_um

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(lags, lag_scores, marker="o")
ax.axvline(best_lag, linestyle="--")
ax.set_xlabel("DMD2 index - DMD1 index, planes")
ax.set_ylabel("median overlap similarity")
ax.set_title(
    f"Best residual lag = {best_lag:+d} planes; "
    f"additional DMD2 offset = {additional_dmd2_z_offset_um:+.2f} um"
)
plt.tight_layout()

print(f"Best residual lag: {best_lag:+d} planes")
print(f"Additional DMD2 z offset: {additional_dmd2_z_offset_um:+.3f} um")
print(f"Recommended total DMD2_Z_OFFSET_UM: {recommended_total_offset:+.3f} um")


## Optional: rerun with overlap fine registration

Use this after the metadata-based warp looks approximately correct. This estimates a residual XY translation of DMD2 relative to DMD1 using the overlap slab.


In [ ]:
FINE_OUT_DIR = OUT_DIR / ("manual_substack_merge_fine" if USE_MANUAL_SUBSTACK_METADATA else "full_reference_merge_fine")

fine_config = Slap2MergeConfig(
    dmd1_tif=DMD1_TIF,
    dmd2_tif=DMD2_TIF,
    out_dir=FINE_OUT_DIR,
    channel=1,
    xy_resolution_um=XY_RESOLUTION_UM,
    z_resolution_um=Z_RESOLUTION_UM,
    z_grid="first",
    dmd2_z_offset_um=DMD2_Z_OFFSET_UM,
    z_interp_method="linear",
    output_compression=None,
    fine_register_overlap=True,
    residual_registration_binning=2,
    residual_upsample_factor=10,
    residual_highpass_sigma_px=8.0,
    residual_max_shift_px=50.0,
    xy_feather_px=32.0,
    z_feather=True,
    write_intermediates=True,
    write_qc_png=True,
)

# Uncomment after the metadata-only run looks reasonable.
# if USE_MANUAL_SUBSTACK_METADATA:
#     fine_summary = merge_dmd_reference_stack_specs(fine_config, spec1, spec2)
# else:
#     fine_summary = merge_dmd_reference_stacks(fine_config)
# print(json.dumps(fine_summary["residual_registration"], indent=2))
# print(json.dumps(fine_summary["outputs"], indent=2))


## CLI equivalents

Full reference stack footprint QC with the measured DMD2 axial offset:

```bash
slap2-align slap2-footprints DMD1-REFERENCE.tif DMD2-REFERENCE.tif out_dir \
  --xy-resolution-um 0.25 \
  --z-resolution-um 1.5 \
  --dmd2-z-offset-um -7.5
```

Full reference stack merge:

```bash
slap2-align slap2-merge-dmds DMD1-REFERENCE.tif DMD2-REFERENCE.tif out_dir \
  --xy-resolution-um 0.25 \
  --z-resolution-um 1.5 \
  --dmd2-z-offset-um -7.5 \
  --fine-register-overlap
```

Use the notebook path for Fiji substacks, because the substack metadata must be manually reconstructed. Set `--dmd2-z-offset-um 0` for a metadata-only baseline.
